Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 150

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import openaire_doi_get_info, _opeanaire_get_subjects, _opeanaire_subjects_to_list

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1896444677', 'title': 'CryoEM BAG for Ile-de-France', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1896444677</identifier>\n    <datestamp>2024-11-13T08:00:09.372Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1896444677</identifier><creators><creator><creatorName nameType="Personal">Hok Sau KWONG

In [7]:
record_titles

['CryoEM BAG for Ile-de-France',
 'Cryo-electron microscopy EPN BAG',
 'Conformational change of chitosan chains upon solvent shifting for nanoprecipitation',
 'Biotransformations of N-heterocyclic carbene gold(I) complexes in cancer cells from speciation studies by X-ray absorption spectroscopy',
 'Origin of charge density waves in the tetragonal Eu(Ga,Al)4 series studied by IXS',
 'Characterization of minimum material requirements for high-speed X-ray diffraction microscopy of operating Surface Acoustic Waves (SAW) devi',
 'Macromolecular crystallography at the Philipps University Marburg (MarBAG)',
 'IBS BAG',
 'Spectro-BCDI of Pt(Pd) alloys during CO2 hydrogenation',
 'Molecular machines in energy transduction, cellular signaling and defense processes',
 'Integrated Ligand Screening Pipelines',
 'Fast Access Beamtime (FAB!) Allocation request',
 'Study of strong cooperative spin-crossover mononuclear complexes using high-resolution X-ray powder diffraction',
 'Uncovering the role o

In [8]:
record_raw_metadata[10]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/21481600</identifier>\n    <datestamp>2018-12-19T14:47:51.614Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-21481600</identifier><creators><creator><creatorName nameType="Personal">Stéphanie MONACO</creatorName><givenName>Stephanie</givenName><familyName>Malbet Monaco</familyName></creator><creator><creatorName nameTy

In [9]:
def ET_from_string(string):
    return ET.ElementTree(ET.fromstring(string))

In [10]:
raw_xml = ET_from_string(record_raw_metadata[13])

In [11]:
raw_xml.getroot()

<Element {http://www.openarchives.org/OAI/2.0/}record at 0x22c05366300>

In [12]:
#print(ET.tostring(raw_xml, pretty_print=True).decode())

In [13]:
# Finding a desired tag anywhere in tree, e.g. title
raw_xml.xpath("//*[local-name()='title']/text()")

['Uncovering the role of water in biomolecular liquid-liquid phase separation']

In [14]:
element  = raw_xml.findall(".//{*}title")
print(element)



print(element[0].text)

[<Element {http://schema.datacite.org/meta/kernel-4.2/}title at 0x22c05467080>]
Uncovering the role of water in biomolecular liquid-liquid phase separation


In [15]:
element  = raw_xml.findall(".//{*}identifier[@identifierType='DOI']")
element
#element[0].text

[<Element {http://schema.datacite.org/meta/kernel-4.2/}identifier at 0x22c05467f40>]

In [16]:
raw_xml.xpath("//*[local-name()='date']/text()")

['2025-09-12T06:00:00Z/2025-09-15T06:00:00Z', '2025-09-15T06:00:00Z']

In [17]:
# getting the DOI of the dataset
# Later, find related DOIs, articles...

# xpath returns a list of strings if /text() is included

raw_xml.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

['10.15151/ESRF-ES-2217189438']

In [18]:
record_number = 6

print(ET.tostring(ET_from_string(record_raw_metadata[record_number - 1]), pretty_print=True).decode())

<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <header>
    <identifier>oai:icatplus.esrf.fr:inv/2264619673</identifier>
    <datestamp>2025-12-08T07:00:23.673Z</datestamp>
  </header>
  <metadata>
    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">
      <schemaVersion>4.2</schemaVersion>
      <datacentreSymbol>EI</datacentreSymbol>
      <payload>
        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2264619673</identifier><creators><creator><creatorName nameType="Personal">Tobias SCHULLI</creatorName><givenName>Tobias</givenName><familyName>Schulli</familyName></creator><creator><creatorName nameType="Personal">Clai

In [19]:
rec = ET_from_string(record_raw_metadata[5])

In [20]:
rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")[0]

'10.15151/ESRF-ES-2264619673'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [21]:
rec = ET_from_string(record_raw_metadata[9])
doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

if doi == []:
    print("NO DOI")
else:
    print(openaire_doi_get_info(doi[0]))

DEBUG:    2026-05-25 14:19:36 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:19:36 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-1299138509 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Jan GEBAUER', 'name': 'Jan', 'surname': 'Gebauer', 'rank': 1, 'pid': None}, {'fullName': 'Montserrat SOLER LOPEZ', 'name': 'Montserrat', 'surname': 'Soler Lopez', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-5924-4734'}, 'provenance': None}}, {'fullName': 'Violetta APPLEGATE', 'name': 'Violetta', 'surname': 'Applegate', 'rank': 3, 'pid': None}, {'fullName': 'Sander SMITS', 'name': 'Sander', 'surname': 'Smits', 'rank': 4, 'pid': None}, {'fullName': 'Yvonne THIELMANN', 'name': 'Yvonne', 'surname': 'Thielmann', 'rank': 5, 'pid': None}, {'fullName': 'Astrid PORT', 'name': 'Astrid', 'surname': 'Port', 'rank': 6, 'pid': {'id': {'scheme': 'orcid_pending', 'valu

In [22]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = ET_from_string(record_raw_metadata[i])
    doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")
    if doi != []:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [23]:
import random
import time

results = []
for i in ix_has_doi:
    rec = ET_from_string(record_raw_metadata[i])
    doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

    if doi == []:
        print(f"{i} : NO DOI")
    else:
        doi = doi[0]

        # small pause to not overwhelm API
        time.sleep(random.uniform(0.01, 0.5))

        print(f"DOI for record {i}")
        try:
            res = openaire_doi_get_info(doi)
        except Exception as e:
            print(f"ERROR: Record {i} met with error: {str(e)}")
            res = {}

        print(res)
        results.append(res)

DOI for record 0
DEBUG:    2026-05-25 14:19:37 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:19:38 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-1896444677 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Hok Sau KWONG', 'name': 'Hok Sau', 'surname': 'Kwong', 'rank': 1, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-8887-0796'}, 'provenance': None}}, {'fullName': 'Marie PREVOST', 'name': 'Marie', 'surname': 'Prevost', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-5621-3708'}, 'provenance': None}}, {'fullName': 'Annalisa MEOLA', 'name': 'Annalisa', 'surname': 'Meola', 'rank': 3, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-2582-4463'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'su

In [24]:
results[6]

{'authors': [{'fullName': 'Gert BANGE',
   'name': 'Gert',
   'surname': 'Bange',
   'rank': 1,
   'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-7826-0932'},
    'provenance': None}},
  {'fullName': 'Igor MELNIKOV',
   'name': 'Igor',
   'surname': 'Melnikov',
   'rank': 2,
   'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-2265-4534'},
    'provenance': None}},
  {'fullName': 'Christopher-Nils MAIS',
   'name': 'Christopher-Nils',
   'surname': 'Mais',
   'rank': 3,
   'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-1927-2885'},
    'provenance': None}}],
 'openAccessColor': None,
 'publiclyFunded': None,
 'type': 'dataset',
 'language': {'code': 'eng', 'label': 'English'},
 'countries': None,
 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'PaN'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID30B'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-2496'}, 'provenance': None},
  {'subje

In [25]:
result_subjects = [_opeanaire_get_subjects(rez) for rez in results]


In [26]:
result_subjects[20]

[{'subject': {'scheme': 'keyword', 'value': 'PaN'}, 'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'Earth Sciences'},
  'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'x-ray probe'},
  'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'Particles, Nuclei and Fields'},
  'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'ID15B'}, 'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'ES-1487'}, 'provenance': None}]

In [27]:
ix = ix_has_doi[20]
record_raw_metadata[ix]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1730964822</identifier>\n    <datestamp>2024-07-21T06:00:23.597Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1730964822</identifier><creators><creator><creatorName nameType="Personal">Angelika ROSA</creatorName><givenName>Angelika</givenName><familyName>Rosa</familyName><nameIdentifier nameIdentifierScheme="ORCID">00

In [28]:
openaire_doi_get_info("10.15151/ESRF-ES-1101291159")

DEBUG:    2026-05-25 14:21:57 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:21:57 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-1101291159 HTTP/1.1" 200 None


{'authors': [{'fullName': 'Wanner, Adrian',
   'name': 'Adrian',
   'surname': 'Wanner',
   'rank': 1,
   'pid': None}],
 'openAccessColor': None,
 'publiclyFunded': None,
 'type': 'dataset',
 'language': {'code': 'eng', 'label': 'English'},
 'countries': None,
 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'Life Sciences'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'LS-3193'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID16A'}, 'provenance': None}],
 'mainTitle': 'Structure and function of neuronal circuits underlying working memory-related sequential activity',
 'subTitle': None,
 'descriptions': ['Working memory is thought to be mediated by stereotypic neuronal activity patterns in the brain such as neural sequences, in which subsets of neurons are sequentially active in a stereotypical temporal order. Neuronal interactions are fundamentally constrained by the synaptic wiring diagram, but the connectivity between neurons t

In [29]:
result_subjects

[[{'subject': {'scheme': 'keyword', 'value': 'PaN'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'electron microscopy'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'CM01'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Particles, Nuclei and Fields'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Macromolecular Crystallography'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-2626'}, 'provenance': None}],
 [{'subject': {'scheme': 'keyword', 'value': 'PaN'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-2366'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'CM01'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Particles, Nuclei and Fields'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-2440'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-2260'}, 'provenan

In [30]:
len(result_subjects)

150

In [31]:
len([result for result in result_subjects if result != []])

144

In [67]:
from utils.enrichment_methods import openaire_doi_get_info, _opeanaire_get_subjects

res = openaire_doi_get_info("10.15151/ESRF-ES-632216926")
print(res)
res_subjects = _opeanaire_get_subjects(res)
print(res_subjects)
res_subjects_list = _opeanaire_subjects_to_list(res_subjects)
print(res_subjects_list)

DEBUG:    2026-05-25 14:26:49 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:26:50 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-632216926 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Amrita PATEL', 'name': 'Amrita', 'surname': 'Patel', 'rank': 1, 'pid': None}, {'fullName': 'Klaus FUTTERER', 'name': 'Klaus', 'surname': 'Futterer', 'rank': 2, 'pid': None}, {'fullName': 'Didier NURIZZO', 'name': 'Didier', 'surname': 'Nurizzo', 'rank': 3, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-7367-5098'}, 'provenance': None}}, {'fullName': 'Daniel PANNE', 'name': 'Daniel', 'surname': 'Panne', 'rank': 4, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-9158-5507'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {

In [66]:
# Mangled, fake DOI sure to get no results:

res = openaire_doi_get_info("123456FALSE")
print(res)
res_subjects = _opeanaire_get_subjects(res)
print(res_subjects)
res_subjects_list = _opeanaire_subjects_to_list(res_subjects)
print(res_subjects_list)

DEBUG:    2026-05-25 14:26:29 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:26:30 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=123456FALSE HTTP/1.1" 200 None
{}
[]
[]


# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword